# 03 Train BayesFlow

Person 3 notebook: train a BayesFlow neural posterior estimator on whitened noisy gravitational-wave strain data.

Expected dataset format:
- `X`: whitened strain signals, shape `(n_simulations, n_samples)`
- `theta`: true parameters, shape `(n_simulations, 6)`
- theta order: `[m1, m2, chi1, chi2, distance, inclination]`

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

DATASET_PATH = PROJECT_ROOT / "data" / "gw_dataset_1000.npz"
MODEL_DIR = PROJECT_ROOT / "models" / "bayesflow_model"
FIGURE_DIR = PROJECT_ROOT / "figures"
FIGURE_DIR.mkdir(exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

DATASET_PATH

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from src.model import (
    PARAMETER_NAMES,
    history_to_losses,
    load_npz_dataset,
    sample_posterior,
    train_workflow,
)

dataset = load_npz_dataset(DATASET_PATH)
print("strain:", dataset["strain"].shape)
print("parameters:", dataset["parameters"].shape)
print("parameter order:", PARAMETER_NAMES)

## Training

Start with a small smoke test (`MAX_SAMPLES = 100` or `1000`). For the final run, set `MAX_SAMPLES = None` and point `DATASET_PATH` to `gw_dataset_10000.npz` if available.

In [ ]:
MAX_SAMPLES = 1000
EPOCHS = 20
BATCH_SIZE = 64

workflow, history, train_data, val_data = train_workflow(
    dataset_path=DATASET_PATH,
    model_dir=MODEL_DIR,
    max_samples=MAX_SAMPLES,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
)

print("train strain:", train_data["strain"].shape)
print("validation strain:", val_data["strain"].shape)

In [ ]:
train_loss, val_loss = history_to_losses(history)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(train_loss, marker="o", label="train loss")
if val_loss is not None:
    ax.plot(val_loss, marker="o", label="validation loss")
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss")
ax.set_title("BayesFlow training loss")
ax.legend()
ax.grid(alpha=0.25)
fig.tight_layout()

loss_path = FIGURE_DIR / "training_loss.png"
fig.savefig(loss_path, dpi=160)
loss_path

## Posterior Example

Draw posterior samples for one held-out validation strain and compare marginal posteriors with the true parameter values.

In [ ]:
test_strain = val_data["strain"][0, :, 0]
true_theta = val_data["parameters"][0]
posterior_samples = sample_posterior(workflow, test_strain, num_samples=1000)

fig, axes = plt.subplots(2, 3, figsize=(12, 6))
axes = axes.ravel()

for i, name in enumerate(PARAMETER_NAMES):
    axes[i].hist(posterior_samples[:, i], bins=35, density=True, alpha=0.75)
    axes[i].axvline(true_theta[i], color="black", linestyle="--", label="true")
    axes[i].set_title(name)
    axes[i].grid(alpha=0.2)

axes[0].legend()
fig.suptitle("Posterior samples for one validation signal")
fig.tight_layout()

posterior_path = FIGURE_DIR / "posterior_example.png"
fig.savefig(posterior_path, dpi=160)
posterior_path

In [ ]:
posterior_mean = posterior_samples.mean(axis=0)
posterior_std = posterior_samples.std(axis=0)
absolute_error = np.abs(posterior_mean - true_theta)

for name, truth, mean, std, err in zip(PARAMETER_NAMES, true_theta, posterior_mean, posterior_std, absolute_error):
    print(f"{name:12s} true={truth:9.4f} posterior_mean={mean:9.4f} std={std:9.4f} abs_error={err:9.4f}")

best = PARAMETER_NAMES[int(np.argmin(absolute_error))]
worst = PARAMETER_NAMES[int(np.argmax(absolute_error))]
print(f"\nShort comment: In this validation example, {best} is closest to the true value and {worst} is least accurate by posterior mean absolute error.")

## Smoke Test Summary

This run validates the full BayesFlow training pipeline end-to-end.

- Dataset: `data/gw_dataset_smoke_20x512.npz`
- Dataset shape: `X = (20, 512)`, `theta = (20, 6)`
- Epochs: `1`
- Batch size: `4`
- Model checkpoint: `models/bayesflow_model/model.keras`
- Figures: `figures/training_loss.png`, `figures/posterior_example.png`

Result: the smoke test completed successfully. This confirms that dataset loading, train/validation splitting, BayesFlow training, checkpointing, and posterior plotting all work. This is a pipeline validation run, not a final scientific training run.
